# Conditional UNet low-N bias: image similarity, summary statistics, and parameter coverage

## tl;dr

This read-only notebook tests three possible contributors to biased low-N cosmology recovery: reproduction of training maps, mismatch in map statistics, and sparse coverage of the requested six-dimensional cosmologies. It compares each conditional UNet with the same held-out CAMELS simulations and joins the saved VGG probe results when available.

The raw conditional artifacts live on Great Lakes. Numerical conclusions remain **pending** until this notebook runs there.

## Context & Methods

Training size N counts 2D HI maps, not independent cosmologies. Pixel cosine is computed after subtracting each map's mean, against **every exact selected training map**. An equal-sized sample of held-out real maps supplies a same-N baseline. Pixel PDFs and FFT power use the same frozen log+tanh model-space normalization for generated, training, and real maps; they are not physical-unit HI statistics. Nearest cosmology uses all six parameters scaled by frozen training-only standard deviations.

Draws from one cosmology are correlated. Probe intervals describe generated-field variation, not Bayesian posterior uncertainty. Use one sweep and one no-guidance sampling protocol at a time.


In [ ]:
import json, os, sys
from pathlib import Path
import numpy as np
import pandas as pd
import yaml
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

candidates = [Path.cwd(), Path.cwd().parent, Path("/home/jiamingp/diffusion_models_repo"),
              Path("/Users/apple/AI/Diffusion_model")]
PROJECT = next(p.resolve() for p in candidates
               if (p/"simdiff_eval"/"conditional_unet_diagnostics.py").is_file())
sys.path.insert(0, str(PROJECT))
from simdiff_eval.conditional_unet_diagnostics import (
    balanced_draws, compare_power, nearest_centered_pixel_cosine, normalize_raw_hi)
from simdiff_eval.parameter_neighbor_control import nearest_parameter_matches

SWEEP = os.environ.get("CONDITIONAL_SWEEP", "nf_conditional_bias_fresh_full_sweep_200k")
MANIFEST = PROJECT/"local"/SWEEP/"manifest.json"
GRID = Path(os.environ.get("CAMELS_HI_GRID",
    "/scratch/huterer_root/huterer0/CAMELS/CMD/3d_grids/IllustrisTNG/"
    "Grids_HI_IllustrisTNG_LH_128_z=0.0.npy"))
SCALES = PROJECT/"local"/"nf_conditional_bias_probe"/"heldout"/"param_norm_stats.json"
RECOVERY_DIR = PROJECT/"results"/SWEEP/"calibration_vgg"
SEED = 123
COSINE_DRAWS_PER_COSMOLOGY = 2
SUMMARY_DRAWS_PER_COSMOLOGY = 8
TRAIN_SUMMARY_LIMIT = 256
plt.rcParams.update({"figure.facecolor":"white", "font.size":11,
                     "axes.spines.top":False, "axes.spines.right":False})
def path(value):
    item = Path(str(value))
    return item if item.is_absolute() else PROJECT/item
display(Markdown(f"Selected sweep: **{SWEEP}**. Manifest: {MANIFEST}"))


## Data

### Check the saved inputs and experiment identity


In [ ]:
rows = json.loads(MANIFEST.read_text()) if MANIFEST.is_file() else []
if rows and (len({r["run_name"] for r in rows}) != len(rows) or
             len({int(r["dataset_size"]) for r in rows}) != len(rows)):
    raise ValueError("Duplicate run or training size in manifest.")
inventory, ready = [], []
for row in sorted(rows, key=lambda r:int(r["dataset_size"])):
    k = int(row.get("heldout_samples_per_cosmology", 64))
    sample = path(row["sample_path"].format(seed=SEED, k=k,
                   sample_label="dpm50", guidance="noguidance"))
    needed = {"sample":sample, "training maps":path(row["prepared_image_path"]),
              "selected slices":path(row["selected_pairs_path"]),
              "training labels":path(row["train_raw_params_path"]),
              "held-out IDs":path(row["heldout_indices_path"]),
              "held-out labels":path(row["heldout_raw_params_path"]),
              "config":path(row["config"]), "parameter scales":SCALES, "CAMELS grid":GRID}
    missing = [name for name, file in needed.items() if not file.is_file()]
    inventory.append({"N maps":int(row["dataset_size"]), "run":row["run_name"],
                      "ready":not missing, "missing":", ".join(missing)})
    if not missing: ready.append((row,sample,k))
if inventory: display(pd.DataFrame(inventory))
else: display(Markdown("**PENDING:** conditional manifest absent locally. Open this notebook in the Great Lakes checkout."))


## Results

### Compute exact image matches, model-space statistics, and six-parameter distances


In [ ]:
similarity_rows, power_rows, pixel_rows, coverage_parts, summary_rows = [], [], [], [], []
gallery = {}
if ready:
    grid = np.load(GRID, mmap_mode="r", allow_pickle=False)
    scales = np.asarray(json.loads(SCALES.read_text())["std"], dtype=float)
    if grid.ndim != 4 or grid.shape[-2:] != (128,128):
        raise ValueError("Unexpected CAMELS HI grid shape.")
    for row, sample_file, k in ready:
        N = int(row["dataset_size"])
        config = yaml.safe_load(path(row["config"]).read_text())
        data = config["data"]
        if config["train"].get("conditioning") != "continuous" or data.get("transform") != ["log"] or data.get("normalization") != "tanh":
            raise ValueError(f"N={N} does not use the expected conditional log+tanh protocol.")
        if path(data["img_path"]).resolve() != path(row["prepared_image_path"]).resolve():
            raise ValueError(f"N={N} config and manifest map paths disagree.")
        norm = data["norm_kwargs"]
        pairs = pd.read_csv(path(row["selected_pairs_path"]))
        train_theta = np.load(path(row["train_raw_params_path"]), allow_pickle=False)
        train = np.load(path(row["prepared_image_path"]), mmap_mode="r", allow_pickle=False)
        heldout = np.atleast_1d(np.loadtxt(path(row["heldout_indices_path"]), dtype=np.int64))
        requested = np.load(path(row["heldout_raw_params_path"]), allow_pickle=False)
        if len(pairs)!=N or len(train)!=N or train_theta.shape!=(N,6) or requested.shape!=(len(heldout),6):
            raise ValueError(f"N={N} selected rows, maps, and parameter labels disagree.")
        if not np.array_equal(pairs.row, np.arange(N)) or pairs.duplicated(["simulation_index","z_index"]).any():
            raise ValueError(f"N={N} selected slice mapping is not unique and ordered.")
        if np.intersect1d(pairs.simulation_index, heldout).size:
            raise ValueError(f"N={N} training simulations overlap held-out simulations.")
        with np.load(sample_file, allow_pickle=False) as archive:
            generated = np.asarray(archive["samples"], dtype=np.float32)
            if (int(archive["samples_per_cosmology"]) != k or
                not np.array_equal(archive["heldout_indices"], heldout) or
                not np.allclose(archive["theta_raw"], requested, rtol=0, atol=1e-6)):
                raise ValueError(f"N={N} saved samples disagree with held-out manifest.")
            if "run_name" in archive and str(archive["run_name"].item()) != row["run_name"]:
                raise ValueError(f"N={N} sample archive names another run.")
            if "dataset_size" in archive and int(archive["dataset_size"]) != N:
                raise ValueError(f"N={N} sample archive names another size.")
            if "seed" in archive and int(archive["seed"]) != SEED:
                raise ValueError(f"N={N} sample archive uses another seed.")
            if "guidance_label" in archive and str(archive["guidance_label"].item()) != "noguidance":
                raise ValueError(f"N={N} sample archive uses guidance.")
        gen_cos, gen_ids, gen_draws = balanced_draws(generated, heldout, k, COSINE_DRAWS_PER_COSMOLOGY)
        z_cos = np.linspace(0, grid.shape[1]-1, COSINE_DRAWS_PER_COSMOLOGY, dtype=int)
        real_cos = normalize_raw_hi(np.stack([grid[int(sim),int(z)] for sim in heldout for z in z_cos]), norm)
        transform = lambda batch: normalize_raw_hi(batch, norm)
        gen_values, gen_nearest = nearest_centered_pixel_cosine(
            gen_cos, train, reference_transform=transform, chunk_size=256)
        real_values, _ = nearest_centered_pixel_cosine(
            real_cos, train, reference_transform=transform, chunk_size=256)
        for sim, draw, value, nearest in zip(gen_ids,gen_draws,gen_values,gen_nearest):
            similarity_rows.append({"N":N,"heldout_sim":int(sim),"kind":"generated",
                "max cosine":float(value),"draw":int(draw),"nearest row":int(nearest)})
        for sim,z,value in zip(np.repeat(heldout,len(z_cos)),np.tile(z_cos,len(heldout)),real_values):
            similarity_rows.append({"N":N,"heldout_sim":int(sim),"kind":"held-out real",
                "max cosine":float(value),"draw":int(z),"nearest row":np.nan})
        near = nearest_parameter_matches(train_theta, pairs.simulation_index.to_numpy(np.int64),
                                         requested, heldout, scales)
        near["N"] = N
        coverage_parts.append(near)
        gen_stats,_,_ = balanced_draws(generated,heldout,k,SUMMARY_DRAWS_PER_COSMOLOGY)
        z_stats = np.linspace(0,grid.shape[1]-1,SUMMARY_DRAWS_PER_COSMOLOGY,dtype=int)
        real_stats = normalize_raw_hi(np.stack([grid[int(sim),int(z)] for sim in heldout for z in z_stats]),norm)
        selected_train = np.linspace(0,N-1,min(N,TRAIN_SUMMARY_LIMIT),dtype=int)
        train_stats = normalize_raw_hi(train[selected_train],norm)
        edges = np.linspace(1,64,26)
        generated_ratio,k_centers = compare_power(gen_stats,real_stats,edges)
        training_ratio,_ = compare_power(train_stats,real_stats,edges)
        for k_center,gr,tr in zip(k_centers,generated_ratio,training_ratio):
            power_rows.append({"N":N,"k":float(k_center),"generated / real":float(gr),
                               "training / real":float(tr)})
        bins=np.linspace(-2,2,101)
        for kind,maps in [("generated",gen_stats),("training",train_stats),("held-out real",real_stats)]:
            counts,edges=np.histogram(maps.ravel(),bins=bins,density=True)
            for center,height in zip((edges[:-1]+edges[1:])/2,counts):
                pixel_rows.append({"N":N,"kind":kind,"pixel":float(center),"density":float(height)})
        summary_rows.append({"N":N,"unique training cosmologies":int(pairs.simulation_index.nunique()),
            "generated maps":len(gen_stats),"real maps":len(real_stats),"training maps":len(train_stats),
            "generated / real low-k":float(np.mean(generated_ratio[:8])),
            "generated / real high-k":float(np.mean(generated_ratio[-8:])),
            "training / real high-k":float(np.mean(training_ratio[-8:])),
            "generated outside histogram":float(np.mean((gen_stats<-2)|(gen_stats>2)))})
        gallery[N]={"generated":gen_cos[:2].copy(),"nearest rows":gen_nearest[:2].copy(),
                    "training path":path(row["prepared_image_path"]),"norm":norm,
                    "cosines":gen_values[:2].copy()}
        print(f"N={N:,}: {len(gen_values)} generated nearest-map queries; {pairs.simulation_index.nunique()} training cosmologies")
similarity=pd.DataFrame(similarity_rows)
power=pd.DataFrame(power_rows)
pixels=pd.DataFrame(pixel_rows)
coverage=pd.concat(coverage_parts,ignore_index=True) if coverage_parts else pd.DataFrame()
summary=pd.DataFrame(summary_rows)
if not summary.empty: display(summary.round(3))
else: display(Markdown("**PENDING:** no run has all required saved inputs here."))


### 1. Generated-to-training cosine versus unseen-real baseline

Every query scans the full exact training subset. Similarity rises as the reference set grows, so compare generated and real maps at the same N. This is centered pixel cosine, not SSCD.


In [ ]:
if not similarity.empty:
    sizes=sorted(similarity.N.unique())
    fig,ax=plt.subplots(figsize=(10,4.5))
    for kind,color,marker,offset in [("generated","#b45b38","o",-.10),
                                      ("held-out real","#315f85","s",.10)]:
        med,lo,hi=[],[],[]
        for N in sizes:
            x=similarity[(similarity.N==N)&(similarity.kind==kind)]["max cosine"].to_numpy()
            q16,q50,q84=np.quantile(x,[.16,.5,.84])
            med.append(q50); lo.append(q50-q16); hi.append(q84-q50)
        ax.errorbar(np.log2(sizes)+offset,med,yerr=[lo,hi],fmt=marker+"-",capsize=3,
                    color=color,label=kind+" (median, 16–84% field spread)")
    ax.set(xlabel="Training maps N",ylabel="Maximum centered pixel cosine",
           title="Similarity to the exact selected training maps")
    ax.set_xticks(np.log2(sizes),[f"{N:,}" for N in sizes])
    ax.grid(alpha=.2); ax.legend(frameon=False,fontsize=9)
    plt.show()
    for N in sorted(set([sizes[0],sizes[-1]])):
        item=gallery[N]
        nearest=normalize_raw_hi(np.load(item["training path"],mmap_mode="r")[item["nearest rows"]],item["norm"])
        generated=item["generated"]
        bounds=np.quantile(np.concatenate([nearest.ravel(),generated.ravel()]),[.01,.99])
        fig,axes=plt.subplots(2,2,figsize=(7.4,7))
        for col in range(2):
            axes[0,col].imshow(generated[col],vmin=bounds[0],vmax=bounds[1],cmap="viridis")
            axes[1,col].imshow(nearest[col],vmin=bounds[0],vmax=bounds[1],cmap="viridis")
            axes[0,col].set_title(f"Generated · cosine {item['cosines'][col]:.3f}")
            axes[1,col].set_title(f"Nearest training row {item['nearest rows'][col]}")
        for axis in axes.ravel(): axis.axis("off")
        fig.suptitle(f"N={N:,}; shared model-space color range")
        fig.tight_layout(); plt.show()


### Optional: reproduce the SSCD feature-cosine comparison

The earlier unconditional plot used SSCD feature cosine. Set SSCD_SIZES (default 128,1024) and RUN_SSCD=1 in the notebook environment to calculate that metric on exact conditional training subsets. The SSCD model must already be cached; no weights are downloaded. The default pixel-cosine diagnostic above is cheaper and remains separately labeled.


In [ ]:
sscd_results=pd.DataFrame()
sscd_path=Path(os.environ.get('SSCD_PATH',str(Path.home()/'.cache'/'torch'/'hub'/'sscd_disc_mixup.torchscript.pt')))
run_sscd=os.environ.get('RUN_SSCD','0')=='1'
sscd_sizes={int(value) for value in os.environ.get('SSCD_SIZES','128,1024').split(',') if value.strip()}
if run_sscd:
    if not sscd_path.is_file(): raise FileNotFoundError(f'SSCD weights not cached: {sscd_path}')
    from simdiff_eval.sscd import load_sscd_torchscript, sscd_embeddings
    device='cuda' if __import__('torch').cuda.is_available() else 'cpu'
    model=load_sscd_torchscript(sscd_path,device=device)
    records=[]
    for row,sample_file,k in ready:
        N=int(row['dataset_size'])
        if N not in sscd_sizes: continue
        norm=yaml.safe_load(path(row['config']).read_text())['data']['norm_kwargs']
        train=np.load(path(row['prepared_image_path']),mmap_mode='r',allow_pickle=False)
        heldout=np.atleast_1d(np.loadtxt(path(row['heldout_indices_path']),dtype=np.int64))
        with np.load(sample_file,allow_pickle=False) as archive:
            gen,ids,_=balanced_draws(archive['samples'],heldout,k,COSINE_DRAWS_PER_COSMOLOGY)
        z=np.linspace(0,grid.shape[1]-1,COSINE_DRAWS_PER_COSMOLOGY,dtype=int)
        real=normalize_raw_hi(np.stack([grid[int(sim),int(slice_id)] for sim in heldout for slice_id in z]),norm)
        options={'device':device,'batch_size':16,'image_size':320,'render_mode':'fixed','value_range':(-1.,1.)}
        gen_features=sscd_embeddings(gen,model,**options).numpy()
        real_features=sscd_embeddings(real,model,**options).numpy()
        reference=[]
        for start in range(0,N,64):
            maps=normalize_raw_hi(train[start:start+64],norm)
            reference.append(sscd_embeddings(maps,model,**options).numpy())
        reference=np.concatenate(reference,axis=0)
        for kind,query in [('generated',gen_features),('held-out real',real_features)]:
            maximum=np.full(len(query),-np.inf)
            for start in range(0,len(reference),1024):
                maximum=np.maximum(maximum,(query@reference[start:start+1024].T).max(axis=1))
            for sim,value in zip(np.repeat(heldout,COSINE_DRAWS_PER_COSMOLOGY),maximum):
                records.append({'N':N,'heldout_sim':int(sim),'kind':kind,'max SSCD cosine':float(value)})
    sscd_results=pd.DataFrame(records)
    if not sscd_results.empty:
        fig,ax=plt.subplots(figsize=(8,4.5))
        for kind,color,marker in [('generated','#b45b38','o'),('held-out real','#315f85','s')]:
            med=sscd_results[sscd_results.kind==kind].groupby('N')['max SSCD cosine'].median()
            ax.plot(np.log2(med.index),med.values,marker=marker,color=color,label=kind)
        ax.set(xlabel='Training maps N',ylabel='Maximum SSCD feature cosine',
               title='Conditional UNet SSCD similarity to exact training maps')
        ax.set_xticks(np.log2(sorted(sscd_results.N.unique())),
                      [f'{N:,}' for N in sorted(sscd_results.N.unique())])
        ax.grid(alpha=.2); ax.legend(frameon=False)
        plt.show()
    else: display(Markdown('**PENDING:** no requested SSCD training sizes have complete saved inputs.'))
else:
    display(Markdown('SSCD calculation is optional and currently off; set RUN_SSCD=1 to reproduce feature-cosine comparisons.'))


### 2. One-point PDF and model-space FFT power

These compare generated maps, exact selected training maps, and real maps at the requested held-out cosmologies. The power heatmaps show the ratio of ensemble mean spectra.


In [ ]:
if not power.empty:
    sizes=sorted(power.N.unique()); k_values=np.sort(power.k.unique())
    fig,axes=plt.subplots(1,2,figsize=(12,max(4.5,.4*len(sizes)+2.5)),sharey=True)
    for ax,column,title in [(axes[0],"training / real","Training / held-out real"),
                             (axes[1],"generated / real","Generated / held-out real")]:
        matrix=power.pivot(index="N",columns="k",values=column).reindex(sizes)
        image=ax.imshow(np.log10(np.clip(matrix.to_numpy(float),1e-6,None)),
                        aspect="auto",origin="lower",cmap="coolwarm",vmin=-.7,vmax=.7)
        ax.set(title=title,xlabel="Radial Fourier k (grid units)")
        ax.set_xticks([0,len(k_values)//2,len(k_values)-1],
                      [f"{k_values[0]:.0f}",f"{k_values[len(k_values)//2]:.0f}",f"{k_values[-1]:.0f}"])
        ax.set_yticks(range(len(sizes)),[f"{N:,}" for N in sizes])
    axes[0].set_ylabel("Training maps N")
    fig.colorbar(image,ax=axes,label="log₁₀ mean power ratio",shrink=.8)
    fig.suptitle("Conditional UNet model-space power",y=1.01)
    plt.show()
if not pixels.empty:
    sizes=sorted(pixels.N.unique()); chosen=sorted(set([sizes[0],sizes[len(sizes)//2],sizes[-1]]))
    fig,axes=plt.subplots(1,len(chosen),figsize=(5*len(chosen),3.5),sharey=True)
    axes=np.atleast_1d(axes)
    for ax,N in zip(axes,chosen):
        for kind,color,style in [("held-out real","#202a33","-"),("training","#aa7c20","--"),
                                  ("generated","#b45b38","-")]:
            part=pixels[(pixels.N==N)&(pixels.kind==kind)]
            ax.plot(part.pixel,part.density,color=color,ls=style,label=kind)
        ax.set(title=f"N={N:,}",xlabel="Normalized pixel value")
        ax.grid(alpha=.15)
    axes[0].set_ylabel("Pixel density"); axes[-1].legend(frameon=False,fontsize=9)
    fig.suptitle("One-point PDF in common log+tanh model space",y=1.04)
    plt.show()


### 3. Is low-N bias related to sparse cosmology coverage?

Distance uses all six parameters. The nearest training label is an oracle coverage control, not a model prediction. The companion parameter-neighbor notebook evaluates the frozen probe on real fields at both cosmologies.


In [ ]:
joined=pd.DataFrame()
if not coverage.empty:
    sizes=sorted(coverage.N.unique())
    fig,axes=plt.subplots(1,2,figsize=(12,4))
    axes[0].boxplot([coverage[coverage.N==N].parameter_distance for N in sizes],
                    positions=np.log2(sizes),widths=.55,showfliers=False)
    axes[0].set(xlabel="Training maps N",ylabel="Nearest six-parameter distance (training SD units)",
                title="Distance to a represented training cosmology")
    axes[0].set_xticks(np.log2(sizes),[f"{N:,}" for N in sizes])
    for N,color in [(sizes[0],"#b45b38"),(sizes[-1],"#315f85")]:
        part=coverage[coverage.N==N]
        axes[1].scatter(part.requested_Omega_m,part.nearest_Omega_m,
                        color=color,label=f"N={N:,}",alpha=.7)
    limits=[min(coverage.requested_Omega_m.min(),coverage.nearest_Omega_m.min()),
            max(coverage.requested_Omega_m.max(),coverage.nearest_Omega_m.max())]
    axes[1].plot(limits,limits,"--",color=".3")
    axes[1].set(xlabel="Requested Ωm",ylabel="Nearest training Ωm",
                title="True parameter offset of nearest cosmology")
    axes[1].legend(frameon=False)
    for ax in axes: ax.grid(alpha=.15)
    plt.show()

recovery_path=RECOVERY_DIR/"bias_probe_per_cosmology_points.csv"
metadata_path=RECOVERY_DIR/"bias_probe_eval_metadata.json"
if recovery_path.is_file() and not coverage.empty:
    recovery=pd.read_csv(recovery_path)
    if "guidance_label" in recovery:
        recovery=recovery[recovery.guidance_label=="noguidance"].copy()
    recovery=recovery[recovery.parameter=="Omega_m"].copy()
    recovery=recovery.rename(columns={"dataset_size":"N"})
    keys=["N","heldout_sim"]
    if recovery.duplicated(keys).any(): raise ValueError("Duplicate no-guidance Ωm recovery points.")
    joined=coverage.merge(recovery[keys+["theta_in","theta_rec_median"]],
                          on=keys,how="inner",validate="one_to_one")
    if len(joined)!=len(coverage): raise ValueError("Recovery points do not cover all ready runs and held-out cosmologies.")
    joined["bias"]=joined.theta_rec_median-joined.theta_in
    joined["nearest offset"]=joined.nearest_Omega_m-joined.theta_in
    generated_cosine=similarity[similarity.kind=="generated"].groupby(keys)["max cosine"].mean()
    joined=joined.join(generated_cosine,on=keys)
    if metadata_path.is_file():
        meta=json.loads(metadata_path.read_text())
        display(Markdown(f"Evaluation probe: **{meta.get('encoder_path','not recorded')}**. Check that all N used this same frozen artifact."))
    chosen=sorted(set([joined.N.min(),joined.N.max()]))
    fig,axes=plt.subplots(1,len(chosen),figsize=(5.8*len(chosen),4),sharey=True)
    axes=np.atleast_1d(axes)
    for ax,N in zip(axes,chosen):
        part=joined[joined.N==N]
        ax.scatter(part.parameter_distance,part.bias,color="#b45b38",label="generated recovery bias")
        ax.scatter(part.parameter_distance,part["nearest offset"],color="#315f85",marker="x",
                   label="nearest true Ωm offset")
        ax.axhline(0,color=".3",ls="--")
        ax.set(title=f"N={int(N):,}; 32 held-out cosmologies",
               xlabel="Nearest six-parameter distance")
        ax.grid(alpha=.15)
    axes[0].set_ylabel("Ωm minus requested Ωm"); axes[-1].legend(frameon=False,fontsize=9)
    fig.suptitle("Coverage versus generated recovery bias",y=1.05)
    plt.show()
    association=[]
    for N,part in joined.groupby("N"):
        association.append({"N":N,"median absolute Ωm bias":np.median(np.abs(part.bias)),
            "Spearman(|bias|, distance)":part.bias.abs().corr(part.parameter_distance,method="spearman"),
            "median generated cosine":part["max cosine"].median()})
    display(pd.DataFrame(association).round(3))
else:
    display(Markdown("**PENDING:** saved VGG recovery points are needed for the bias-versus-distance check."))


### 4. If available, compare the same frozen probe on real fields

The companion parameter-neighbor evaluator applies the frozen probe to real held-out fields and fields from the nearest represented training cosmology. This comparison helps separate probe error from generator error. Only matching run names are used.


In [ ]:
CONTROL_DIR=Path(os.environ.get('NEIGHBOR_CONTROL_DIR',str(PROJECT/'results'/'parameter_neighbor_control_v1')))
control_file=CONTROL_DIR/'control_points.csv'
if control_file.is_file() and not joined.empty:
    controls=pd.read_csv(control_file)
    matching_runs={row['run_name'] for row,_,_ in ready}
    controls=controls[(controls.run_name.isin(matching_runs)) &
                      (controls.parameter=='Omega_m') &
                      (controls.kind.isin(['heldout_real','nearest_training_field']))].copy()
    if controls.empty:
        display(Markdown('**PENDING:** the real-field control table has no matching run names.'))
    else:
        controls=controls.rename(columns={'dataset_size':'N'})
        controls['absolute residual']=(controls.theta_rec_median-controls.theta_in).abs()
        generated_summary=joined.assign(**{'absolute residual':joined.bias.abs()})[['N','heldout_sim','absolute residual']]
        generated_summary['kind']='generated'
        combined=pd.concat([controls[['N','heldout_sim','kind','absolute residual']],generated_summary],ignore_index=True)
        counts=combined.groupby(['N','kind']).heldout_sim.nunique()
        if (counts!=32).any(): raise ValueError('Real-field controls do not contain all 32 held-out cosmologies per run.')
        aggregate=combined.groupby(['N','kind'])['absolute residual'].median().unstack('kind')
        display(aggregate.round(4))
        fig,ax=plt.subplots(figsize=(9,4.5))
        for kind,color,marker in [('heldout_real','#315f85','s'),
                                  ('nearest_training_field','#aa7c20','^'),
                                  ('generated','#b45b38','o')]:
            if kind in aggregate:
                ax.plot(np.log2(aggregate.index),aggregate[kind],marker=marker,color=color,label=kind)
        ax.set(xlabel='Training maps N',ylabel='Median absolute Ωm residual',
               title='Same frozen probe on generated and real fields')
        ax.set_xticks(np.log2(aggregate.index),[f'{int(N):,}' for N in aggregate.index])
        ax.grid(alpha=.2); ax.legend(frameon=False)
        plt.show()
else:
    display(Markdown('**PENDING:** real-field probe controls have not been produced for this conditional sweep.'))


## Takeaways

Interpret the plots together. Generated maps that are closer to training maps than held-out real maps suggest reproduction. A mismatch already present between the selected training maps and held-out real maps shows the empirical target is limited at that N; an additional generated mismatch suggests generator or sampling error. Large six-parameter distance with a nearest-label offset in the same direction as probe recovery makes sparse coverage plausible. Compare the same frozen probe on held-out real fields before assigning the remaining bias to the generator.

An association across 32 held-out cosmologies is diagnostic, not causal proof. No training, sampling, or job submission occurs in this notebook.
